<a href="https://colab.research.google.com/github/harrisonm23-byte/HuggingFaceML1/blob/claude%2Fhuggingface-ml-research-setup-ulvgve/notebooks/12_the_mountain_gemma3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 12 — The mountain on Gemma 3, recording what happens inside the model

The same mountain as notebook 11 (one Deutsche Bank desk, the five days of the case, the chats arriving on their real dates, memory carried by the desk diary, unreadable replies sent back to correct), run on **Gemma 3** in Colab instead of Gemma 4 through the API, so we can look inside the model.

**What is recorded inside the model.** At every decision (every 15-minute update), the model's internal state at each layer: the residual stream at the last token of the update, just before it starts writing its reply, and its average over the reply it writes. These are saved with each day, for the next notebook to read through the **Gemma Scope 2** sparse autoencoders (`google/gemma-scope-2-4b-it`, every layer; they turn the activations into features with readable descriptions on Neuronpedia). The questions they can answer: what is active when the ask arrives versus the same moment with no ask, under loss versus gain, and with "you"; and whether anything like "manipulation" or "this is wrong" is active when the model goes along anyway.

**Settings below.** Gemma 3 4B by default: it fits a T4 at full precision, the precision Gemma Scope was trained on, and is about 3-4 times faster than 12B. The first batch puts the takeover effect first: the three rungs of the ladder (the real trader's lines are the agent's own for 5, 3 or 1 days) on the real chats, plus the no-ask control, one run each: 6 runs, about 3 hours on a T4 (about 30 minutes a run). The rungs differ only on the days where Deutsche Bank's real lines are handed over or not (12 Jan, 1 Apr, 5 Aug), so the same days compare with and without the inherited history. Then the framed versions (`VERSIONS = ALL_VERSIONS`); finished days are skipped. A second open model with public SAEs: Llama 3.1 8B (Llama Scope).

**Saving.** Everything goes to Google Drive (`silver_study/mountain_<model>/`) as each day ends: `days.jsonl` (each day: steps, trades, replies, diary, measures, in the same format as the API runs) and `acts/` (one file per day with the activations, about 8 MB a day for 4B). If Colab disconnects, run all again: it continues from the next day. When done, send Claude `days.jsonl` (the activations stay on Drive for the next notebook).

**To run:** Runtime → Change runtime type → T4 GPU; add your Hugging Face token as the Colab secret `HF_TOKEN` (key icon) and accept the Gemma 3 licence on Hugging Face; then Runtime → Run all, and allow Google Drive.

In [ ]:
# Install the libraries and log in to Hugging Face (reads the HF_TOKEN secret)
!pip install -q -U transformers accelerate bitsandbytes
import os, sys, json, hashlib, time
from huggingface_hub import login
try:
    from google.colab import userdata
    login(token=userdata.get('HF_TOKEN'))
except Exception:
    login(token=os.environ.get("HF_TOKEN"))

# Get the simulator, the price files and the framed chats from the repo
if os.path.exists("../sim/mountain.py"):                    # opened inside the repo (Jupyter)
    REPO = os.path.abspath("..")
else:
    if not os.path.exists("HuggingFaceML1"):
        os.system("git clone -q -b claude/huggingface-ml-research-setup-ulvgve https://github.com/harrisonm23-byte/HuggingFaceML1.git")
    else:
        os.system("git -C HuggingFaceML1 pull -q")
    REPO = os.path.abspath("HuggingFaceML1")
os.chdir(REPO)
sys.path.insert(0, os.path.join(REPO, "sim"))
from mountain import DAYS, MODES as ALL_MODES, VERSIONS as ALL_VERSIONS, load_chat_setups, make_day, one_line, opening

In [ ]:
# Settings
MODEL = "google/gemma-3-4b-it"       # also "google/gemma-3-12b-it" (Gemma Scope 2) or "meta-llama/Llama-3.1-8B-Instruct"
                                     # (Llama Scope; accept its licence on Hugging Face); both load in 4-bit on a T4, slower
VERSIONS = ["real", "no ask"]        # first batch: the takeover ladder on the real chats, and the no-ask control;
                                     # then the framed versions: ALL_VERSIONS (real, loss, gain, loss + you, gain + you, no ask)
MODES = ALL_MODES                    # the ladder: takeover 5 days, 3 days, 1 day, one rung per run in turn (keep this order)
RUNS = 3                             # runs per version; run k is on rung MODES[k % len(MODES)], as in the API runs
RECENT = 4                           # earlier steps today shown in full (older ones as one line each), as in the API runs
TEMPERATURE = 0.7
MAX_NEW_TOKENS = 300
SAVE_ACTIVATIONS = True
SAVE_TO_DRIVE = True

## The model, with a recorder on every layer

In [ ]:
import torch, numpy as np
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline

tok = AutoTokenizer.from_pretrained(MODEL)
if any(size in MODEL.lower() for size in ("8b", "12b", "27b")):     # too big for a T4 at full precision
    from transformers import BitsAndBytesConfig
    quant = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.bfloat16)
    mod = AutoModelForCausalLM.from_pretrained(MODEL, device_map="auto", quantization_config=quant)
else:
    mod = AutoModelForCausalLM.from_pretrained(MODEL, device_map="auto", dtype=torch.bfloat16)
chat = pipeline("text-generation", model=mod, tokenizer=tok)

# The recorder: on every decoder layer, keep the residual stream at the last token of the update (when the model has
# read everything and is about to reply) and add up its value over each token it writes (for the average)
LAYERS = [m for n, m in mod.named_modules() if type(m).__name__.endswith("DecoderLayer")]
rec = {}

def reset_recorder():
    rec.clear()
    rec.update({"last": [None] * len(LAYERS), "sum": [None] * len(LAYERS), "n": 0})

def make_hook(k):
    def hook(module, inputs, output):
        h = output[0] if isinstance(output, tuple) else output
        if h.shape[1] > 1:                       # the whole update at once: keep its last token
            rec["last"][k] = h[0, -1].float()
        else:                                    # one token of the reply
            rec["sum"][k] = h[0, -1].float() if rec["sum"][k] is None else rec["sum"][k] + h[0, -1].float()
            if k == 0:
                rec["n"] += 1
    return hook

for k, layer in enumerate(LAYERS):
    layer.register_forward_hook(make_hook(k))

def small(x):
    # float16 to keep the files small, unless a value is too large for it (then float32)
    h = x.astype(np.float16)
    return h if np.isfinite(h).all() else x

def recorded():
    # (layers, width) arrays: the state before the reply, and its average over the reply
    last = torch.stack(rec["last"]).cpu().numpy()
    mean = (torch.stack(rec["sum"]) / max(rec["n"], 1)).cpu().numpy() if rec["n"] else np.zeros_like(last)
    return small(last), small(mean)

print("Loaded", MODEL, "|", len(LAYERS), "layers, width", mod.config.get_text_config().hidden_size)

## The agent

The same instructions and memory as the API runs (`scripts/run_mountain_api.py`): the opening, the desk diary of earlier days, one line per earlier step today, the last four steps in full, and the new update. A reply sent back for correction is followed by what was wrong, and the model answers again; the recording kept for a step is the one for the reply that was used.

In [ ]:
OPENING = opening()

def make_agent(diary, store):
    def agent(observation, d):
        turns = [{"role": "system", "content": OPENING}]
        older, recent = d.log[:-RECENT] if len(d.log) > RECENT else [], d.log[-RECENT:]
        memory = (["Desk diary (your earlier days):\n" + "\n".join(diary)] if diary else []) + \
                 (["Earlier today:\n" + "\n".join(one_line(r) for r in older)] if older else [])
        if memory:
            turns += [{"role": "user", "content": "\n\n".join(memory)}, {"role": "assistant", "content": "Noted."}]
        for r in recent:
            turns += [{"role": "user", "content": r["observation"]}, {"role": "assistant", "content": r["reply"] or "(no reply)"}]
        turns.append({"role": "user", "content": observation})
        for bad, note in d.corrections:         # sent back for correction: the rejected reply, then what was wrong
            turns += [{"role": "assistant", "content": bad or "(no reply)"}, {"role": "user", "content": note}]
        reset_recorder()
        out = chat(turns, max_new_tokens=MAX_NEW_TOKENS, do_sample=True, temperature=TEMPERATURE)
        reply = out[0]["generated_text"][-1]["content"].strip()
        store["acts"] = recorded()
        store["finish"] = "MAX_TOKENS" if len(tok(reply)["input_ids"]) >= MAX_NEW_TOKENS else "STOP"
        return reply
    return agent

## Run the days

Each run is one version and one rung, the five days in order; the diary of the days already saved is carried into the next. Each day is saved to Drive as it ends, and finished days are skipped if you run all again.

In [ ]:
OUT = "."
if SAVE_TO_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        OUT = "/content/drive/MyDrive/silver_study"
    except Exception as e:
        print("Drive not available, saving in Colab only:", e)
OUT = os.path.join(OUT, "mountain_" + MODEL.split("/")[-1])
os.makedirs(os.path.join(OUT, "acts"), exist_ok=True)
SAVE = os.path.join(OUT, "days.jsonl")

# The same setup ID as the API runs, so days from a different version of the code are never mixed in
SET_ID = "mt" + hashlib.md5(json.dumps([open(f).read() for f in ("sim/mountain.py", "sim/chat_market.py", "sim/silver_day.py",
                                                                  "data/silver/chat_market_pilot.json")] + [RECENT]).encode()).hexdigest()[:8]
SETUPS = load_chat_setups()
FRAMED = json.load(open("data/silver/framed_chats_75.json"))["results"]
rows = [json.loads(l) for l in open(SAVE)] if os.path.exists(SAVE) else []
assert all(r["set"] == SET_ID and r["model"] == MODEL for r in rows), f"{SAVE} holds days from other code or another model: use another folder"
todo = [(v, MODES[k % len(MODES)], k, date) for k in range(RUNS) for v in VERSIONS for date in DAYS
        if not any((r["version"], r["run"], r["date"]) == (v, k, date) for r in rows)]
print(f"{len(rows)} days saved, {len(todo)} to go | {SET_ID}")

t0 = time.time()
for n, (v, mo, k, date) in enumerate(todo, 1):
    diary = [r["diary"] for r in sorted(rows, key=lambda r: r["date"]) if (r["version"], r["run"]) == (v, k) and r["date"] < date]
    day = make_day(date, v, FRAMED, SETUPS, mode=mo)
    store, acts, finishes = {}, [], []
    agent = make_agent(diary, store)
    while not day.done:
        day.step(agent)
        acts.append(store["acts"])
        finishes.append(store["finish"])
    steps = []
    for r, finish in zip(day.log, finishes):
        st = {x: r.get(x) for x in ("time", "price_seen", "observation", "reply", "status", "problem", "target", "traded", "chat",
                                    "quote", "joined", "rival_traded", "stop_fills", "delivered", "price_after", "valued_at",
                                    "position", "over_limit", "pnl", "corrections")}
        st["finish"] = finish
        steps.append(st)
    name = f"{v.replace(' + ', '_plus_').replace(' ', '_')}_r{k}_{mo.replace(' ', '_')}_{date}"
    if SAVE_ACTIVATIONS:
        np.savez_compressed(os.path.join(OUT, "acts", name + ".npz"), last=np.stack([a[0] for a in acts]),
                            reply_mean=np.stack([a[1] for a in acts]), times=np.array([r["time"] for r in day.log]))
    row = {"version": v, "run": k, "mode": mo, "date": date, "model": MODEL, "set": SET_ID, "summary": day.summary(),
           "diary": day.diary(), "finishes": finishes, "memory_days": len(diary), "steps": steps,
           "acts_file": f"acts/{name}.npz" if SAVE_ACTIVATIONS else None}
    with open(SAVE, "a") as f:
        f.write(json.dumps(row) + "\n")
    rows.append(row)
    s = row["summary"]
    joined = [c for c, m in s["chats"].items() if m["joined"]]
    el = time.time() - t0
    print(f"{n}/{len(todo)} | {v} | {mo} | run {k + 1} | {date} | P&L ${s['pnl']:+,.0f} | joined {joined or 'none'} | "
          f"stops early {sum(f['early'] for f in s['stops_fired'])} | {el / 60:.0f} min, about {el / n * (len(todo) - n) / 60:.0f} min to go")

## The results so far

The same tables as the API runs (written by `scripts/run_mountain_api.py --page-only` into the Drive folder: `results.md`, and `runs.md` with every day step by step).

In [ ]:
os.system(f'python scripts/run_mountain_api.py --page-only --model {MODEL} --runs {RUNS} --modes "{",".join(MODES)}" '
          f'--versions "{",".join(VERSIONS)}" --out "{OUT}"')
print(open(os.path.join(OUT, "results.md")).read())
try:
    from google.colab import files
    files.download(SAVE)
except Exception:
    pass

Send Claude `days.jsonl` (downloaded above, and on Drive). Before saving this notebook to GitHub, clear the outputs (Edit → Clear all outputs).